In [1]:
pip install vaderSentiment transformers torch scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 2.0 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [ ]:
"""
Phase 4: Sentiment classifier - model comparison + application to the 38K set.
Run in a Kaggle notebook (GPU recommended, needs internet to download model weights).
 
Step A - Compare 4 pretrained models against the 3800-record reference set's gold
          labels (column: 'relative sentiments' - negative / neutral only, no
          positive examples present - see WARNING printed below).
Step B - Apply the winning model to the full 38K review column.
 
Output of Step B is a lean table: review_id, predicted_sentiment, confidence_score.
brand_name / car_year / Rating are pulled from the main 38K file directly during
the Phase 5 merge step, not duplicated here - single source of truth for metadata.
 
Requires (Kaggle): pip install vaderSentiment transformers torch scikit-learn tqdm

"""

In [2]:
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
)
from tqdm import tqdm
from transformers import pipeline
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [3]:
# ---------------- CONFIG (edit to match your files) ----------------
REF_PATH = "/kaggle/input/datasets/raghavenderg/car-review-chatbot/3700_review.csv"
MAIN_PATH = "/kaggle/input/datasets/raghavenderg/car-review-chatbot/38000_processed.csv"
REVIEW_COL = "review"
LABEL_COL = "relative sentiments"   # confirmed real column name in the reference set
ID_COL = "review_id"
BATCH_SIZE = 64                     # Kaggle GPU can handle larger batches than CPU
# ---------------------------------------------------------------------

device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

Using device: GPU


In [4]:
ref = pd.read_csv(REF_PATH)
if ID_COL not in ref.columns:
    ref[ID_COL] = ref.index
gold = ref[LABEL_COL].str.lower().str.strip()
 
class_counts = gold.value_counts()
print("Gold label distribution:\n", class_counts)
if "positive" not in class_counts.index:
    print("\nWARNING: no 'positive' examples in the reference set. No candidate's "
          "positive-class performance can be validated here. Phase 5's merge script "
          "adds a Rating-based proxy check for that instead.")
 
majority_baseline = class_counts.max() / class_counts.sum()
print(f"Majority-class baseline accuracy: {majority_baseline:.3f} "
      f"(a model must clear this to be worth anything)")
 
reviews_ref = ref[REVIEW_COL].fillna("").astype(str).tolist()

Gold label distribution:
 relative sentiments
negative    3323
neutral      377
Name: count, dtype: int64

Majority-class baseline accuracy: 0.898 (a model must clear this to be worth anything)


In [5]:
# ============ STEP A: COMPARE CANDIDATE MODELS ============
comparison_rows = []
 
def evaluate(name, pred_labels):
    acc = accuracy_score(gold, pred_labels)
    macro_f1 = f1_score(gold, pred_labels, average="macro", zero_division=0)
    bal_acc = balanced_accuracy_score(gold, pred_labels)
    comparison_rows.append(
        {"model": name, "accuracy": acc, "macro_f1": macro_f1, "balanced_accuracy": bal_acc}
    )
    print(f"\n--- {name} ---")
    print(f"Accuracy: {acc:.3f} | Macro-F1: {macro_f1:.3f} | Balanced accuracy: {bal_acc:.3f}")
    print(classification_report(gold, pred_labels, zero_division=0))
 
 
LABEL_MAP = {"label_0": "negative", "label_1": "neutral", "label_2": "positive"}
 
 
def normalize_label(raw_label):
    """Maps LABEL_0/1/2-style outputs to text; passes through already-text labels."""
    return LABEL_MAP.get(str(raw_label).lower(), str(raw_label).lower())

In [6]:
# --- Candidate 1: VADER (rule-based baseline, no GPU needed) ---
vader = SentimentIntensityAnalyzer()
 
 
def vader_predict(text):
    score = vader.polarity_scores(str(text))["compound"]
    if score >= 0.05:
        return "positive"
    if score <= -0.05:
        return "negative"
    return "neutral"
 
 
vader_labels = [vader_predict(t) for t in reviews_ref]
evaluate("VADER", vader_labels)
 
# --- Candidate 2: DistilBERT-SST2 (binary transformer, movie-review domain) ---
distilbert = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device, truncation=True, max_length=512,
)
distilbert_labels = [normalize_label(r["label"]) for r in distilbert(reviews_ref, batch_size=BATCH_SIZE)]
evaluate("DistilBERT-SST2 (binary)", distilbert_labels)
 
# --- Candidate 3: cardiffnlp RoBERTa (3-class, handles neutral, informal text) ---
roberta = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=device, truncation=True, max_length=512,
)
roberta_labels = [normalize_label(r["label"]) for r in roberta(reviews_ref, batch_size=BATCH_SIZE)]
evaluate("RoBERTa-3class (cardiffnlp)", roberta_labels)
 
# --- Candidate 4: SiEBERT (binary, trained across many diverse domains - often more
#     robust than SST2-only models, worth testing even though it can't do neutral) ---
siebert = pipeline(
    "sentiment-analysis",
    model="siebert/sentiment-roberta-large-english",
    device=device, truncation=True, max_length=512,
)
siebert_labels = [normalize_label(r["label"]) for r in siebert(reviews_ref, batch_size=BATCH_SIZE)]
evaluate("SiEBERT (binary, cross-domain)", siebert_labels)

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



--- VADER ---
Accuracy: 0.995 | Macro-F1: 0.661 | Balanced accuracy: 0.990
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00      3323
     neutral       0.99      0.98      0.99       377
    positive       0.00      0.00      0.00         0

    accuracy                           0.99      3700
   macro avg       0.66      0.66      0.66      3700
weighted avg       1.00      0.99      1.00      3700



config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]


--- SiEBERT (binary, cross-domain) ---
Accuracy: 0.830 | Macro-F1: 0.310 | Balanced accuracy: 0.462
              precision    recall  f1-score   support

    negative       0.93      0.92      0.93      3323
     neutral       0.00      0.00      0.00       377
    positive       0.00      0.00      0.00         0

    accuracy                           0.83      3700
   macro avg       0.31      0.31      0.31      3700
weighted avg       0.84      0.83      0.83      3700



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [7]:
# ---- Comparison summary ----
comparison_df = pd.DataFrame(comparison_rows).sort_values("macro_f1", ascending=False)
comparison_df.to_csv("../data/interim/phase4_model_comparison.csv", index=False)
print("\n=== Model comparison (sorted by macro-F1) ===")
print(comparison_df.to_string(index=False))
print("\nNOTE: the binary models (DistilBERT, SiEBERT) cannot predict 'neutral' and are "
      "structurally penalized on that ~10% of the gold set. If the 3-class model wins, "
      "this is part of why - not just raw sentiment accuracy.")
 
# ============ PICK THE WINNER ============
# Update these three lines after reading phase4_model_comparison.csv above.
WINNING_MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
winner_pipeline = roberta
print(f"\nUsing '{WINNING_MODEL_NAME}' for Step B. Change winner_pipeline above "
      f"if a different candidate scored higher on your data.")


=== Model comparison (sorted by macro-F1) ===
                         model  accuracy  macro_f1  balanced_accuracy
                         VADER  0.994595  0.660893           0.989936
   RoBERTa-3class (cardiffnlp)  0.857027  0.470984           0.746386
      DistilBERT-SST2 (binary)  0.891351  0.319249           0.496238
SiEBERT (binary, cross-domain)  0.830270  0.309553           0.462233

NOTE: the binary models (DistilBERT, SiEBERT) cannot predict 'neutral' and are structurally penalized on that ~10% of the gold set. If the 3-class model wins, this is part of why - not just raw sentiment accuracy.

Using 'cardiffnlp/twitter-roberta-base-sentiment-latest' for Step B. Change winner_pipeline above if a different candidate scored higher on your data.


In [8]:
import pandas as pd
from transformers.pipelines.pt_utils import KeyDataset
from datasets import Dataset
from tqdm.auto import tqdm

# ============ STEP B: APPLY WINNER TO THE FULL 38K SET ============

df = pd.read_csv(MAIN_PATH)
if ID_COL not in df.columns:
    df[ID_COL] = df.index

# Ensure strings
df[REVIEW_COL] = df[REVIEW_COL].fillna("").astype(str)

# Convert Pandas DataFrame to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df[[REVIEW_COL]])

predicted_labels = []
confidence_scores = []

# Pass KeyDataset directly to the pipeline with batch_size specified
for out in tqdm(winner_pipeline(KeyDataset(hf_dataset, REVIEW_COL), batch_size=BATCH_SIZE), total=len(df)):
    predicted_labels.append(normalize_label(out["label"]))
    confidence_scores.append(out["score"])

sentiment_output = pd.DataFrame({
    ID_COL: df[ID_COL],
    "predicted_sentiment": predicted_labels,
    "confidence_score": confidence_scores,
})

sentiment_output.to_csv("/kaggle/working/phase4_sentiment_predictions_38k.csv", index=False)
print(f"\nDone. {len(sentiment_output)} predictions saved to phase4_sentiment_predictions_38k.csv")
print(sentiment_output.head())

  0%|          | 0/31938 [00:00<?, ?it/s]


Done. 31938 predictions saved to phase4_sentiment_predictions_38k.csv
   review_id predicted_sentiment  confidence_score
0          0            positive          0.942460
1          1            positive          0.987694
2          2            positive          0.955537
3          3            positive          0.967720
4          4            positive          0.932461
